In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

print("All libraries imported successfully!")

In [ ]:
import os

print(os.listdir("/kaggle/working"))


In [ ]:
url = "https://books.toscrape.com/"

response = requests.get(url, timeout=20)

print("Status Code:", response.status_code)
print("Website Connected:", response.status_code == 200)

In [ ]:
import re

rating_map = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}
books = []

soup = BeautifulSoup(response.text, "html.parser")

for book in soup.select("article.product_pod"):
    title = book.h3.a["title"]
    price_text = book.select_one("p.price_color").text
    price = float(re.sub(r'[^0-9.]', '', price_text))
    rating = rating_map[book.select_one("p.star-rating")["class"][1]]
    books.append({"Title": title, "Price": price, "Rating": rating})

df = pd.DataFrame(books)
print("df created!")
print(df.head())
print(df.shape)

In [ ]:
df = pd.DataFrame(books)

print("Dataset created successfully!")

display(df.head(10))

In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

In [ ]:
df.to_csv("books_dataset.csv", index=False)

print("Dataset saved successfully!")

In [ ]:
all_books = []

for page in range(1, 51):

    if page == 1:
        page_url = "https://books.toscrape.com/"
    else:
        page_url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    page_response = requests.get(page_url)

    page_soup = BeautifulSoup(page_response.text, "html.parser")

    book_items = page_soup.find_all("article", class_="product_pod")

    for book in book_items:

        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        availability = book.find(
            "p", class_="instock availability"
        ).text.strip()
        rating = book.p["class"][1]

        all_books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability
        })

print("Total books collected:", len(all_books))

In [ ]:
df = pd.DataFrame(all_books)

display(df.head(10))

print("Dataset Shape:", df.shape)

In [ ]:
df.to_csv("all_books_dataset.csv", index=False)

print("Complete dataset saved successfully!")

In [ ]:
print(df.isnull().sum())

In [ ]:
print("Duplicate records:", df.duplicated().sum())

In [ ]:

import pandas as pd
import numpy as np

df["Price"] = pd.to_numeric(
    df["Price"].astype(str).str.replace(r'[^0-9.\-]', '', regex=True),
    errors='coerce'
)

print(df["Price"].head())

In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["Rating"] = df["Rating"].map(rating_map)

print(df["Rating"].head())

In [ ]:
df.info()
display(df.head(10))

In [ ]:
print("BOOK DATA ANALYSIS")
print("--------------------")

print("Average Book Price:", round(df["Price"].mean(), 2))

print("Highest Book Price:", df["Price"].max())

print("Lowest Book Price:", df["Price"].min())

print("Average Book Rating:", round(df["Rating"].mean(), 2))

print("Most Common Rating:", df["Rating"].mode()[0])

In [ ]:
rating_counts = df["Rating"].value_counts().sort_index()

print(rating_counts)

In [ ]:
rating_counts.plot(
    kind="bar",
    figsize=(8, 5),
    title="Book Rating Distribution"
)

plt.xlabel("Star Rating")
plt.ylabel("Number of Books")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df["Price"], bins=20)

plt.title("Distribution of Book Prices")
plt.xlabel("Price (£)")
plt.ylabel("Number of Books")

plt.show()

In [ ]:
!pip install -q nltk

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from collections import Counter

nltk.download("stopwords")

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_title(title):
    title = title.lower()
    words = re.findall(r'\b[a-z]{2,}\b', title)

    words = [
        word for word in words
        if word not in stop_words
    ]

    return words

df["Cleaned_Title"] = df["Title"].apply(clean_title)

display(df[["Title", "Cleaned_Title"]].head(10))

In [ ]:
all_words = []

for words in df["Cleaned_Title"]:
    all_words.extend(words)

word_counts = Counter(all_words)

print("Top 10 Most Common Words:")

print(word_counts.most_common(10))

In [ ]:
top_words = word_counts.most_common(10)

words = [item[0] for item in top_words]
counts = [item[1] for item in top_words]

plt.figure(figsize=(10, 5))

plt.bar(words, counts)

plt.title("Most Frequently Used Words in Book Titles")
plt.xlabel("Words")
plt.ylabel("Frequency")

plt.xticks(rotation=45)
plt.show()

In [ ]:
df.to_csv("cleaned_books_nlp.csv", index=False)

print("Final cleaned dataset saved!")
print("Total records:", len(df))
print("Total columns:", len(df.columns))

In [1]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print("Folder:", root)
    print("Files:", files)

Folder: /kaggle/input
Files: []
Folder: /kaggle/input/datasets
Files: []
Folder: /kaggle/input/datasets/fat33maali
Files: []
Folder: /kaggle/input/datasets/fat33maali/book-data-collection-and-nlp-analysis
Files: ['all_books_dataset.csv', 'books_dataset.csv', 'cleaned_books_nlp.csv']


In [1]:
import pandas as pd

file_path = "/kaggle/input/datasets/fat33maali/book-data-collection-and-nlp-analysis/cleaned_books_nlp.csv"

df_check = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset Shape:", df_check.shape)
print("Columns:", df_check.columns.tolist())

display(df_check.head())

Dataset loaded successfully!
Dataset Shape: (1000, 5)
Columns: ['Title', 'Price', 'Rating', 'Availability', 'Cleaned_Title']


,Title,Price,Rating,Availability,Cleaned_Title
0,A Light in the Attic,51.77,3,In stock,"['light', 'attic']"
1,Tipping the Velvet,53.74,1,In stock,"['tipping', 'velvet']"
2,Soumission,50.10,1,In stock,['soumission']
3,Sharp Objects,47.82,4,In stock,"['sharp', 'objects']"
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,"['sapiens', 'brief', 'history', 'humankind']"
